# Getting Started With the SynthPopCan Beginner API

This notebook mirrors the [Getting Started With the Beginner API](https://synthpopcan.readthedocs.io/en/latest/library-getting-started.html) documentation page.

The beginner API is the supported first path for notebooks, teaching examples, and short scripts. It gives you a few functions for common work:

- fit seed rows to control totals with IPF;
- save weighted or expanded IPF output;
- generate linked household/person rows from a prepared model package;
- calibrate generated linked household/person candidates to small-area household controls.

**Before you start:** run the install cell below once per environment, then restart the kernel.

In [ ]:
# Install SynthPopCan — run once per environment, then restart the kernel.
%pip install synthpopcan

## First Notebook Cell

Start with the path helper and the SynthPopCan import:

In [ ]:
from pathlib import Path

import synthpopcan as spc

## Fit Seed Rows With IPF

A notebook is a good place to inspect files, try a small fit, and record the choices that shaped the output.

The example below uses the small fixture files bundled with the SynthPopCan source checkout. If you are running outside a checkout, substitute paths to your own seed and control CSVs.

In [ ]:
# Paths to the example fixture files.
# Replace these with your own seed and control CSVs if running outside
# the SynthPopCan source checkout.
fixture_root = Path("tests/fixtures/workflows/microdata_ipf")
seed_path = fixture_root / "expected-seed.csv"
controls_path = fixture_root / "controls.csv"

In [ ]:
# Read a seed file and inspect its shape before fitting.
# The first element gives the row count; the second shows one row.
seed = spc.read_seed(seed_path)

len(seed), seed[0]

In [ ]:
# List the column names from the first row.
sorted(seed[0])

In [ ]:
# Read controls and inspect the margins.
# Each margin has a name, a list of dimensions, and a set of cells.
controls = spc.read_controls(controls_path)

[(margin.name, margin.dimensions, len(margin.cells)) for margin in controls.margins]

In [ ]:
# Fit the seed rows to the controls.
# Check converged=True before treating the output as finished.
fit = spc.fit_ipf(
    seed,
    controls,
    weight_field="WEIGHT",
    max_iterations=250,
    tolerance=0.01,
)

{
    "converged": fit.converged,
    "iterations": fit.iterations,
    "max_abs_error": fit.max_abs_error,
}

In [ ]:
# Write the fitted weights once the fit is acceptable.
spc.write_weights(fit, "synthetic-weights.csv")

In [ ]:
# Expand to integer rows when a downstream tool needs one row per record.
# Keep the weighted file for auditing — only expand small examples.
expanded = spc.expand_population(fit)

len(expanded), expanded[0]

In [ ]:
spc.write_population(expanded, "expanded-population.csv")

## Work Directly From Paths

When you do not need to inspect or filter rows between steps, pass paths directly:

In [ ]:
fit = spc.fit_ipf(seed_path, controls_path, weight_field="WEIGHT")
spc.write_weights(fit, "weights.csv")

Use in-memory objects when you want to inspect or modify data between steps.
Add a Markdown cell above any filter explaining why the selection was made and what it excludes.

In [ ]:
seed = spc.read_seed(seed_path)
controls = spc.read_controls(controls_path)

adult_seed = [row for row in seed if row["AGEGRP"] == "adult"]
len(adult_seed)

## Generate From a Prepared Model Package

The beginner API treats model training and release packaging as advanced preparation work.
Once a package has been prepared and reviewed, generation is short.

Replace `"linked-model-package.json"` with the path to your own reviewed package.

In [ ]:
package = spc.read_model_package("linked-model-package.json")

population = spc.generate_from_model(
    package,
    households=100,
    conditions={"geo": "Demo North"},
    random_seed=42,
)

len(population.households), len(population.persons)

In [ ]:
# Write linked output to a directory.
# Creates synthetic-linked-population/households.csv and persons.csv.
spc.write_population(population, "synthetic-linked-population")

## Assign Generated Rows To Small Areas

Small-area synthesis starts after a candidate linked population exists.
The controls must include one geography dimension (such as `ct` for census tract)
plus household dimensions already present in the candidate household CSV.

In [ ]:
summary = spc.calibrate_small_area_linked(
    households="candidate-households.csv",
    persons="candidate-persons.csv",
    controls="ct-tenure-controls.csv",
    geography_dimension="ct",
    geography_column="ct",
    households_out="synthetic-households.csv",
    persons_out="synthetic-persons.csv",
    report_out="small-area-report.json",
)

summary["assigned_households"], summary["assigned_persons"]

## Reproducible Generation

Use a fixed random seed when generating from a model package so notebook runs are reproducible:

In [ ]:
population = spc.generate_from_model(
    "linked-model-package.json",
    households=250,
    random_seed=2026,
)

In [ ]:
# Use require_publishable=False only for local development or teaching.
# Publishable or shared work should use reviewed packages.
population = spc.generate_from_model(
    "linked-model-package.json",
    households=25,
    require_publishable=False,
)

## A Good Notebook Record

For humanities-facing research, a useful notebook should read like a short method note.
Include Markdown cells that answer:

- What source files or model packages did we use?
- What geography, period, or population is included?
- What controls were fitted, and which were left out?
- Did the IPF fit converge?
- Did we keep weighted output or expand it?
- What should another reader not infer from this output?

The code cells should then support that narrative. If a result changes after you rerun the notebook, the prose should make it clear which random seed, filters, controls, or package version shaped the result.